In [ ]:
import logging
from datetime import datetime
from datasets import load_dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses,
)
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator, SequentialEvaluator, SimilarityFunction

# 1. Configurações e Modelo
logging.basicConfig(format="%(asctime)s - %(message)s", level=logging.INFO)

model_name = "neuralmind/bert-large-portuguese-cased"
matryoshka_dims = [1024, 512, 256, 128, 64]
batch_size = 32 

model = SentenceTransformer(model_name)

# --- Funções de Normalização ---
def normalize_sts(example):
    # STSb e ASSIN usam escala 0-5. CoSENTLoss e Evaluators preferem 0-1 ou escala original bem definida.
    # Vamos normalizar para 0-1 para consistência.
    score_col = "similarity_score" if "similarity_score" in example else "relatedness_score"
    return {"label": float(example[score_col]) / 5.0}

# 2. Carga e Preparação dos Datasets
train_dataset = {}
eval_dataset = {}

# --- NLI: MultipleNegativesRankingLoss (Melhor para Pares Positivos/Inferência) ---
nli_splits = ["pt_anli", "pt_fever", "pt_ling", "pt_mnli", "pt_wanli"]
for split_name in nli_splits:
    ds = load_dataset("MoritzLaurer/multilingual-NLI-26lang-2mil7", split=split_name)
    # Filtramos apenas Entailment (label 0) para agir como pares positivos no MNRL
    train_dataset[f"nli_{split_name}"] = ds.filter(lambda x: x["label"] == 0).rename_columns({
        "premise": "anchor", "hypothesis": "positive"
    }).select_columns(["anchor", "positive"])

# --- STSb Multi MT (Subset 'pt'): CoSENTLoss (Melhor para Scores Contínuos) ---
stsb_pt = load_dataset("stsb_multi_mt", "pt")
train_dataset["stsb"] = stsb_pt["train"].map(normalize_sts).select_columns(["sentence1", "sentence2", "label"])
eval_dataset["stsb"] = stsb_pt["dev"].map(normalize_sts).select_columns(["sentence1", "sentence2", "label"])

# --- ASSIN 1: CoSENTLoss ---
assin1 = load_dataset("assin", split="train")
train_dataset["assin1"] = assin1.map(normalize_sts).rename_columns({
    "premise": "sentence1", "hypothesis": "sentence2"
}).select_columns(["sentence1", "sentence2", "label"])

# --- ASSIN 2: CoSENTLoss ---
assin2 = load_dataset("assin2")
train_dataset["assin2"] = assin2["train"].map(normalize_sts).rename_columns({
    "premise": "sentence1", "hypothesis": "sentence2"
}).select_columns(["sentence1", "sentence2", "label"])
eval_dataset["assin2"] = assin2["validation"].map(normalize_sts).rename_columns({
    "premise": "sentence1", "hypothesis": "sentence2"
}).select_columns(["sentence1", "sentence2", "label"])

# 3. Definição das Losses com Matryoshka
base_mnrl_loss = losses.MultipleNegativesRankingLoss(model)
matryoshka_mnrl_loss = losses.MatryoshkaLoss(model, base_mnrl_loss, matryoshka_dims=matryoshka_dims)

base_cosent_loss = losses.CoSENTLoss(model)
matryoshka_cosent_loss = losses.MatryoshkaLoss(model, base_cosent_loss, matryoshka_dims=matryoshka_dims)

# Mapeamento dinâmico corrigido
loss_map = {}

# Mapear datasets de TREINO
for name in train_dataset.keys():
    if "nli" in name:
        loss_map[name] = matryoshka_mnrl_loss
    else:
        loss_map[name] = matryoshka_cosent_loss

# 4. Evaluator (STSb e ASSIN2 Dev)
evaluators = []
for dim in matryoshka_dims:
    # STSb Dev
    evaluators.append(EmbeddingSimilarityEvaluator(
        sentences1=stsb_pt["dev"]["sentence1"],
        sentences2=stsb_pt["dev"]["sentence2"],
        scores=[s / 5.0 for s in stsb_pt["dev"]["similarity_score"]],
        name=f"stsb-pt-dev-{dim}",
        truncate_dim=dim,
    ))
    # ASSIN2 Dev
    evaluators.append(EmbeddingSimilarityEvaluator(
        sentences1=assin2["validation"]["premise"],
        sentences2=assin2["validation"]["hypothesis"],
        scores=[s / 5.0 for s in assin2["validation"]['relatedness_score']],
        name=f"assin2-dev-{dim}",
        truncate_dim=dim,
    ))

dev_evaluator = SequentialEvaluator(evaluators)

# 5. Argumentos de Treinamento
args = SentenceTransformerTrainingArguments(
    output_dir=f"output/bertimbau-large-matryoshka-hybrid-loss",
    num_train_epochs=5,
    per_device_train_batch_size=batch_size,
    warmup_steps=0.1,
    fp16=True,
    eval_strategy="steps",
    eval_steps=250,
    save_strategy="steps",
    logging_steps=100,
    learning_rate=1e-5,
    gradient_accumulation_steps=32,
    multi_dataset_batch_sampler="proportional",
)

# 6. Trainer
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=loss_map,
    evaluator=dev_evaluator,
)

trainer.train()

# 7. Avaliação Final (Test Sets)
test_evaluators = []
for dim in matryoshka_dims:
    # STSb Test
    test_evaluators.append(EmbeddingSimilarityEvaluator(
        sentences1=stsb_pt["test"]["sentence1"],
        sentences2=stsb_pt["test"]["sentence2"],
        scores=[s / 5.0 for s in stsb_pt["test"]["similarity_score"]],
        name=f"stsb-test-{dim}",
        truncate_dim=dim,
    ))
    # ASSIN2 Test
    test_evaluators.append(EmbeddingSimilarityEvaluator(
        sentences1=assin2["test"]["premise"],
        sentences2=assin2["test"]["hypothesis"],
        scores=[s / 5.0 for s in assin2["test"]['relatedness_score']],
        name=f"assin2-test-{dim}",
        truncate_dim=dim,
    ))

final_test_evaluator = SequentialEvaluator(test_evaluators)
final_test_evaluator(model)

model.save_pretrained("models/bertimbau-large-portuguese-matryoshka-hybrid")

2026-03-20 00:13:24,914 - Use pytorch device_name: mps
2026-03-20 00:13:24,915 - Load pretrained SentenceTransformer: neuralmind/bert-large-portuguese-cased
2026-03-20 00:13:25,081 - HTTP Request: HEAD https://huggingface.co/neuralmind/bert-large-portuguese-cased/resolve/main/modules.json "HTTP/1.1 404 Not Found"
2026-03-20 00:13:25,083 - No sentence-transformers model found with name neuralmind/bert-large-portuguese-cased. Creating a new one with mean pooling.
2026-03-20 00:13:25,232 - HTTP Request: HEAD https://huggingface.co/neuralmind/bert-large-portuguese-cased/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
2026-03-20 00:13:25,371 - HTTP Request: HEAD https://huggingface.co/neuralmind/bert-large-portuguese-cased/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-20 00:13:25,383 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/neuralmind/bert-large-portuguese-cased/aa302f6ea73b759f7df9cad58bd272127b67ec28/config.json "HTTP/1.1 200 OK"


In [23]:
train_dataset['stsb'].to_pandas()

,sentence1,sentence2,label
0,Um avião está a descolar.,Um avião aéreo está a descolar.,1.00
1,Um homem está a tocar uma grande flauta.,Um homem está a tocar uma flauta.,0.76
2,Um homem está a espalhar queijo desfiado numa ...,Um homem está a espalhar queijo desfiado sobre...,0.76
3,Três homens estão a jogar xadrez.,Dois homens estão a jogar xadrez.,0.52
4,Um homem está a tocar violoncelo.,Um homem sentado está a tocar violoncelo.,0.85
...,...,...,...
5744,Gales Graves com a Tempestade Clodagh a atingi...,Merkel promete a solidariedade da OTAN para co...,0.00
5745,Dezenas de reféns egípcios feitos reféns por t...,O número de mortes por acidente de barco egípc...,0.00
5746,Presidente em direcção ao Bahrein,Presidente Xi: China para continuar a ajudar a...,0.00
5747,"A China, Índia prometem estabelecer mais laços...",A China mexe para assegurar os negociantes de ...,0.00
